# Power Transformation Experiment Results

## Objective

Evaluate the impact of applying **Power Transformation (Yeo-Johnson)** to different combinations of numerical features while keeping the remaining preprocessing pipeline unchanged.

## Top 10 Results

| Rank | Power Transformed Feature(s) | Best Model | Test AP | Train AP | Validation AP |
|-----:|------------------------------|------------|------:|---------:|--------------:|
| 1 | Duration, Net Sales | LogisticRegression (Elastic Net) | 0.107269 | 0.086275 | **0.086035** |
| 2 | Duration, Commission, Net Sales | LogisticRegression (Elastic Net) | 0.106789 | 0.086198 | 0.085890 |
| 3 | Duration, Commission, Net Sales | LogisticRegression (Lasso) | 0.105306 | 0.086800 | 0.085811 |
| 4 | Duration, Net Sales | LogisticRegression (Lasso) | 0.105236 | 0.086821 | 0.085811 |
| 5 | Duration, Net Sales | LogisticRegression (Ridge) | 0.107102 | 0.086485 | 0.085792 |
| 6 | Net Sales | LogisticRegression (Elastic Net) | 0.105219 | 0.086298 | 0.085649 |
| 7 | Net Sales | LogisticRegression (Ridge) | 0.104912 | 0.086539 | 0.085547 |
| 8 | Duration, Net Sales | LogisticRegression (Base) | 0.104644 | 0.087210 | 0.085498 |
| 9 | Commission, Net Sales | LogisticRegression (Elastic Net) | 0.104818 | 0.086332 | 0.085485 |
| 10 | Duration, Net Sales, Age | LogisticRegression (Elastic Net) | **0.108317** | 0.085920 | 0.085485 |

## Conclusion

- Applying **Power Transformation (Yeo-Johnson)** to **Duration** and **Net Sales** achieved the highest **validation Average Precision (0.086035)**.
- Although transforming **Duration, Net Sales, and Age** produced the highest cross-validation Average Precision, it did **not** produced the best Test performance, indicating that the additional transformation did not improve generalization.
- Therefore, the final preprocessing applies **Power Transformation (Yeo-Johnson) to Duration and Net Sales**, while the remaining numerical features continue to use the standard preprocessing pipeline.

In [50]:
import warnings
warnings.filterwarnings("ignore")

In [51]:
from pathlib import Path
import sys

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root))

In [52]:
READ_CSV="../../data/interim/data_travel_insurance_interim.csv"
RANDOM_STATE=42
TARGET_TRANSFORM_COLS = ["Destination"]
SAVE_RESULT = "results/feature_encoding_log_transform_optimization_results.csv"

In [53]:
from src.utils import benchmark_models

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder, TargetEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import PowerTransformer
from scipy.stats import shapiro

from itertools import combinations

from xgboost import XGBClassifier

from feature_engine.outliers import Winsorizer

In [54]:
df = pd.read_csv(READ_CSV)
df.head()

,Agency,Agency Type,Distribution Channel,Product Name,Duration,Destination,Net Sales,Commission,Age,Claim,Is Refund,Suspected Fraud,Commission Rate
0,C2B,Airlines,Online,Annual Silver Plan,365,SINGAPORE,216.0,54.0,57,0,No,No,0.25
1,EPX,Travel Agency,Online,Cancellation Plan,4,MALAYSIA,10.0,0.0,33,0,No,No,0.00
2,JZI,Airlines,Online,Basic Plan,19,INDIA,22.0,7.7,26,0,No,No,0.35
3,EPX,Travel Agency,Online,2 way Comprehensive Plan,20,UNITED STATES,112.0,0.0,59,0,No,No,0.00
4,C2B,Airlines,Online,Bronze Plan,8,SINGAPORE,16.0,4.0,28,0,No,No,0.25


In [55]:
x = df.drop(columns=["Claim"])
y = df["Claim"]


In [56]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [ ]:
NUMERIC_COLS = [features for features in x_train.columns if x_train[features].dtypes != "O"]
CATEGORICAL_COLS = [features for features in x_train.columns if x_train[features].dtypes == "O"]

In [58]:
numeric_pipeline = Pipeline([
    ("winsorizer_iqr", Winsorizer(capping_method="iqr", fold=1.5)),
    ("RobustScaler", RobustScaler()),
])

numeric_log_pipeline = Pipeline([
    ("power", PowerTransformer(method="yeo-johnson")),
    ("RobustScaler", RobustScaler()),
])

categorical_ohe_pipeline = Pipeline([
     ("OneHotEncoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"))
])

categorical_target_pipeline = Pipeline([
     ("TargetEncoder", TargetEncoder())
])

In [61]:
logreg_base = LogisticRegression(random_state=RANDOM_STATE)
logreg_lasso = LogisticRegression(penalty="l1", solver="liblinear", random_state=RANDOM_STATE)
logreg_ridge = LogisticRegression(penalty="l2", solver="saga", random_state=RANDOM_STATE)
logreg_elasticnet = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, random_state=RANDOM_STATE)
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
knn = KNeighborsClassifier()
rf = RandomForestClassifier(random_state=RANDOM_STATE)
ab = AdaBoostClassifier(random_state=RANDOM_STATE)
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
xgb = XGBClassifier(random_state=RANDOM_STATE)


In [62]:
skewness_results = (
    x_train[NUMERIC_COLS]
    .skew()
    .abs()
    .rename("Absolute Skewness")
    .reset_index()
)

skewness_results.columns = ["Feature", "Absolute Skewness"]

skewness_results = skewness_results.sort_values(
    by="Absolute Skewness",
    ascending=False
)

In [63]:
LOG_TRANSFORM_COLS = (
    skewness_results.loc[
        skewness_results["Absolute Skewness"] > 1,
        "Feature"
    ]
    .tolist()
)

print("Log Transform Candidates:", LOG_TRANSFORM_COLS)

Log Transform Candidates: ['Duration', 'Commission', 'Net Sales', 'Age']


In [64]:
models = {
    "LogisticRegressionBase": logreg_base,
    "LogisticRegressionLasso": logreg_lasso,
    "LogisticRegressionRidge": logreg_ridge,
    "LogisticRegressionElasticNet": logreg_elasticnet,
    "DecisionTree": dt,
    "KNearestNeigbor": knn,
    "RandomForest": rf,
    "AdaBoost": ab,
    "GradientBoost": gb,
    "XGBoost": xgb
}


log_cols = [[]]  

for r in range(1, len(LOG_TRANSFORM_COLS) + 1):
    log_cols.extend(combinations(LOG_TRANSFORM_COLS, r))

all_results = []

for cols in log_cols:

    cols = list(cols)
    normal_cols = [c for c in NUMERIC_COLS if c not in cols]

    transformers = []

    if normal_cols:
        transformers.append(
            ("numeric_pipeline", numeric_pipeline, normal_cols)
        )

    if cols:
        transformers.append(
            ("numeric_log_pipeline", numeric_log_pipeline, cols)
        )

    transformers.extend([
        (
            "categorical_ohe_pipeline",
            categorical_ohe_pipeline,
            [c for c in CATEGORICAL_COLS if c not in TARGET_TRANSFORM_COLS],
        ),
        (
            "categorical_target_pipeline",
            categorical_target_pipeline,
            TARGET_TRANSFORM_COLS,
        ),
    ])

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression())
    ])

    experiment_results = benchmark_models(
        pipeline,
        models,
        x_train,
        y_train,
        x_test,
        y_test,
        5,
        "Base"
    )

    experiment_results["Power Transformed"] = (
        "None"
        if len(cols) == 0
        else ", ".join(cols)
    )

    all_results.append(experiment_results)

results = pd.concat(all_results, ignore_index=True)

In [65]:
results = results.sort_values("mean_ap_validate_score", ascending=False)
results[0:10]

,name,ap_test_score,mean_ap_train_score,mean_ap_validate_score,f1_test_score,recall_test_score,precision_test_score,accuracy_test_score,Power Transformed
60,LogisticRegressionElasticNet_Base,0.107269,0.086275,0.086035,0.0,0.0,0.0,0.98282,"Duration, Net Sales"
110,LogisticRegressionElasticNet_Base,0.106789,0.086198,0.085890,0.0,0.0,0.0,0.98282,"Duration, Commission, Net Sales"
111,LogisticRegressionLasso_Base,0.105306,0.086800,0.085811,0.0,0.0,0.0,0.98282,"Duration, Commission, Net Sales"
61,LogisticRegressionLasso_Base,0.105236,0.086821,0.085811,0.0,0.0,0.0,0.98282,"Duration, Net Sales"
62,LogisticRegressionRidge_Base,0.107102,0.086485,0.085792,0.0,0.0,0.0,0.98282,"Duration, Net Sales"
30,LogisticRegressionElasticNet_Base,0.105219,0.086298,0.085649,0.0,0.0,0.0,0.98282,Net Sales
31,LogisticRegressionRidge_Base,0.104912,0.086539,0.085547,0.0,0.0,0.0,0.98282,Net Sales
63,LogisticRegressionBase_Base,0.104644,0.087210,0.085498,0.0,0.0,0.0,0.98282,"Duration, Net Sales"
80,LogisticRegressionElasticNet_Base,0.104818,0.086332,0.085485,0.0,0.0,0.0,0.98282,"Commission, Net Sales"
130,LogisticRegressionElasticNet_Base,0.108317,0.085920,0.085485,0.0,0.0,0.0,0.98282,"Duration, Net Sales, Age"


In [66]:
results.to_csv(SAVE_RESULT, index=False)